In [1]:
import numpy as np
datapath = "../../Preprocessing/Encoded_Data/mopb_embeddings.npz"
data = np.load(datapath)
embeddings = data["embeddings"]
labels = data["labels"]

In [2]:
mid_layer  = embeddings[:, 0, :]
last_layer = embeddings[:, 1, :]
embeddings_pooled = embeddings.mean(axis=1)
embeddings_pooled.shape

(1300, 320)

In [3]:
from sklearn.metrics import v_measure_score
from sklearn.cluster import HDBSCAN

embedding_options = {
    "Mid layer": mid_layer,
    "Last layer": last_layer,
    "Pooled": embeddings_pooled
}

best_per_embedding = {}
global_best_score = 0
global_best = {}

for embedding_name, embedding_data in embedding_options.items():
    best_score = 0
    best_params = {}
    best_results = {}

    for min_cluster_size in [5, 10, 20, 30, 50]:
        for min_samples in [1, 5, 10]:
            for metric in ["euclidean", "manhattan", "cosine"]:

                clusterer = HDBSCAN(
                    min_cluster_size=min_cluster_size,
                    min_samples=min_samples,
                    metric=metric,
                    copy=False
                )

                hdb_labels = clusterer.fit_predict(embedding_data)

                if len(set(hdb_labels)) <= 1:  # all noise or one cluster
                    continue

                score = v_measure_score(labels, hdb_labels)

                # best for THIS embedding
                if score > best_score:
                    best_score = score
                    best_params = {
                        'min_cluster_size': min_cluster_size,
                        'min_samples': min_samples,
                        'metric': metric
                    }
                    best_results = {
                        'num_clusters': len(set(hdb_labels)) - (1 if -1 in hdb_labels else 0),
                        'num_noise': sum(hdb_labels == -1)
                    }

                # global best across all embeddings
                if score > global_best_score:
                    global_best_score = score
                    global_best = {
                        'embedding': embedding_name,
                        'min_cluster_size': min_cluster_size,
                        'min_samples': min_samples,
                        'metric': metric
                    }

    best_per_embedding[embedding_name] = {
        'score': best_score,
        'params': best_params,
        'results': best_results
    }

print("Best per embedding:")
for k, v in best_per_embedding.items():
    print(k, v)

print("\nGlobal best:")
print(global_best_score, global_best)

Best per embedding:
Mid layer {'score': 0.5126530406785322, 'params': {'min_cluster_size': 20, 'min_samples': 1, 'metric': 'manhattan'}, 'results': {'num_clusters': 5, 'num_noise': np.int64(155)}}
Last layer {'score': 0.6102575564165705, 'params': {'min_cluster_size': 20, 'min_samples': 1, 'metric': 'cosine'}, 'results': {'num_clusters': 7, 'num_noise': np.int64(149)}}
Pooled {'score': 0.6008220695267511, 'params': {'min_cluster_size': 20, 'min_samples': 1, 'metric': 'cosine'}, 'results': {'num_clusters': 7, 'num_noise': np.int64(205)}}

Global best:
0.6102575564165705 {'embedding': 'Last layer', 'min_cluster_size': 20, 'min_samples': 1, 'metric': 'cosine'}


### Soft predict for no -1 label

In [4]:
import hdbscan

embedding_options = {
    "Mid layer": mid_layer,
    "Last layer": last_layer,
    "Pooled": embeddings_pooled
}

best_per_embedding = {}
global_best_score = 0
global_best = {}

for embedding_name, embedding_data in embedding_options.items():
    best_score = 0
    best_params = {}
    best_results = {}

    for min_cluster_size in [5, 10, 20, 30, 50]:
        for min_samples in [1, 5, 10]:
            for metric in ["euclidean", "manhattan"]:

                clusterer = hdbscan.HDBSCAN(
                    min_cluster_size=min_cluster_size,
                    min_samples=min_samples,
                    metric=metric,
                    prediction_data=True
                )

                predictions = clusterer.fit_predict(embedding_data)

                soft_clusters = hdbscan.all_points_membership_vectors(clusterer)
                best_cluster = np.argmax(soft_clusters, axis=1)
                no_noise_predictions = np.where(predictions == -1, best_cluster, predictions)

                if len(set(no_noise_predictions)) <= 1:  # all noise or one cluster
                    continue

                score = v_measure_score(labels, no_noise_predictions)

                if score > best_score:
                    best_score = score
                    best_params = {
                        'min_cluster_size': min_cluster_size,
                        'min_samples': min_samples,
                        'metric': metric
                    }
                    best_results = {
                        'num_clusters': len(set(no_noise_predictions)) - (1 if -1 in no_noise_predictions else 0),
                        'num_noise': sum(no_noise_predictions == -1)
                    }

                if score > global_best_score:
                    global_best_score = score
                    global_best = {
                        'embedding': embedding_name,
                        'min_cluster_size': min_cluster_size,
                        'min_samples': min_samples,
                        'metric': metric
                    }

    best_per_embedding[embedding_name] = {
        'score': best_score,
        'params': best_params,
        'results': best_results
    }

print("Best per embedding:")
for k, v in best_per_embedding.items():
    print(k, v)

print("\nGlobal best:")
print(global_best_score, global_best)

Best per embedding:
Mid layer {'score': 0.5560065025148552, 'params': {'min_cluster_size': 20, 'min_samples': 1, 'metric': 'manhattan'}, 'results': {'num_clusters': 5, 'num_noise': np.int64(0)}}
Last layer {'score': 0.6697693724534473, 'params': {'min_cluster_size': 30, 'min_samples': 1, 'metric': 'manhattan'}, 'results': {'num_clusters': 7, 'num_noise': np.int64(0)}}
Pooled {'score': 0.6240347027407839, 'params': {'min_cluster_size': 50, 'min_samples': 1, 'metric': 'euclidean'}, 'results': {'num_clusters': 6, 'num_noise': np.int64(0)}}

Global best:
0.6697693724534473 {'embedding': 'Last layer', 'min_cluster_size': 30, 'min_samples': 1, 'metric': 'manhattan'}
